Shield, WAF, and Cognito are AWS's services for protecting applications from network attacks and managing user identity. Shield defends against DDoS attacks. WAF filters malicious HTTP traffic at the application layer. Cognito handles user authentication and authorisation for web and mobile applications. Together they form the security perimeter between the internet and your application.

## AWS Shield

Shield protects AWS resources from **DDoS** (Distributed Denial of Service) attacks — attempts to overwhelm your infrastructure with malicious traffic.

### Shield Standard
- **Free** — automatically enabled for all AWS customers
- Protects against common **Layer 3 and Layer 4** attacks: SYN floods, UDP reflection attacks, volumetric attacks
- Active on: CloudFront, Route 53, Elastic Load Balancers, EC2 (via Elastic IP)
- No configuration required — always on

### Shield Advanced
- **$3,000/month** per organisation + data transfer fees
- All Shield Standard protections, plus:
  - **Layer 7 DDoS protection** (application layer) — for HTTP floods against CloudFront, ALB, API Gateway
  - **DDoS cost protection** — AWS credits any scaling costs incurred during a DDoS attack (EC2, ELB, CloudFront, Route 53)
  - **24/7 access to AWS Shield Response Team (SRT)** — AWS experts assist in real time during an attack
  - **Real-time attack visibility** — detailed metrics and CloudWatch alarms
  - **Automatic WAF rule creation** — SRT can create and deploy WAF rules on your behalf during attacks
  - **Global threat dashboard** — visibility into attack patterns across AWS
  - Protected resources: EC2, ELB, CloudFront, Route 53, Global Accelerator, Elastic IP

| | Shield Standard | Shield Advanced |
|---|---|---|
| Cost | Free | $3,000/month per org |
| L3/L4 protection | Yes | Yes |
| L7 protection | No | Yes |
| SRT access | No | Yes (24/7) |
| Cost protection | No | Yes |
| Attack visibility | Basic | Detailed metrics + alarms |

> **Exam tip:** Shield Advanced is the answer when a scenario mentions DDoS cost protection, SRT access, or Layer 7 DDoS protection.

## AWS WAF — Web Application Firewall

WAF filters HTTP/HTTPS traffic at **Layer 7** based on rules you define. It blocks common web exploits before they reach your application.

### Where WAF deploys
WAF attaches to:
- **CloudFront** (global — rules evaluated at edge)
- **Application Load Balancer** (regional)
- **API Gateway** (regional)
- **AppSync** (GraphQL APIs)
- **Cognito User Pool**

### Web ACL
A **Web ACL** (Access Control List) is the top-level WAF resource — a container for rules with a default action (Allow or Block).

```text
Incoming request
        │
        ▼
  Web ACL rules (evaluated in priority order)
  ├── Rule 1: Block IPs in bad-ip-set          → BLOCK
  ├── Rule 2: Rate limit > 2000 req/5min       → BLOCK
  ├── Rule 3: AWS Managed Rules (SQLi, XSS)    → BLOCK
  └── Default action                           → ALLOW
```

### Rule types

| Rule | What it matches |
|---|---|
| **IP set** | Block or allow specific IP addresses or CIDR ranges |
| **Geo match** | Block or allow by country |
| **Rate-based** | Block IPs that exceed a request rate threshold (per 5-minute window) |
| **String match** | Match on URI, query string, headers, body (exact, prefix, contains, regex) |
| **SQL injection** | Detect SQL injection patterns in request fields |
| **XSS** | Detect cross-site scripting patterns |
| **Size constraint** | Block requests with oversized body, headers, or URI |
| **Managed rule group** | Pre-built rule sets maintained by AWS or Marketplace vendors |

### AWS Managed Rule Groups
Pre-built, regularly updated rule sets — no tuning required:
- **Core Rule Set (CRS)** — protects against OWASP Top 10
- **Known Bad Inputs** — blocks known attack patterns
- **SQL Database** — SQL injection protection
- **Linux / POSIX / Windows** — OS-specific exploits
- **Bot Control** — detect and block bots, scrapers, crawlers
- **Account Takeover Prevention** — detect credential stuffing on login pages
- **Anonymous IP List** — block Tor exit nodes, VPN providers, proxies

### Rule actions
- **Allow** — pass the request through
- **Block** — return HTTP 403
- **Count** — count the match without blocking (useful for testing rules)
- **CAPTCHA** — challenge the user with a CAPTCHA

### WAF Logging
Log all WAF decisions to: Kinesis Data Firehose → S3 (for analysis), CloudWatch Logs, or S3 directly.

## AWS Firewall Manager

Firewall Manager centrally manages WAF rules, Shield Advanced protections, Security Groups, and Network Firewall policies **across all accounts in an AWS Organisation**.

```text
Management account
  └── Firewall Manager policy
        ├── Auto-applies WAF rules to all ALBs in all member accounts
        ├── Enforces Shield Advanced on all CloudFront distributions
        └── Reports compliance across the organisation
```

- New accounts joining the organisation automatically get the policies applied
- Single pane of glass for security posture across hundreds of accounts

> **Exam tip:** "Apply security rules across all accounts in an organisation" or "centralise WAF management" → Firewall Manager.

## Amazon Cognito

Cognito provides **user authentication and authorisation** for web and mobile applications. It has two distinct components with very different purposes: **User Pools** and **Identity Pools**.

```text
User Pools:      Who are you?       (authentication — username/password, social login)
Identity Pools:  What can you do?   (authorisation — exchange token for AWS credentials)
```

## Cognito User Pools

A **User Pool** is a managed user directory. It handles sign-up, sign-in, and user management for your application.

### What User Pools provide
- **User registration and sign-in** — username/email + password
- **Social identity providers** — sign in with Google, Facebook, Apple, Amazon
- **Enterprise federation** — SAML 2.0 and OIDC identity providers (Okta, Azure AD, ADFS)
- **MFA** — SMS, TOTP (e.g., Google Authenticator)
- **Password policies** — complexity, rotation
- **Email and phone verification**
- **Pre/post authentication Lambda triggers** — custom logic at auth events
- **Hosted UI** — pre-built, customisable sign-in page

### Authentication flow
```text
User signs in
      │
      ▼
Cognito User Pool authenticates
      │
      ▼
Returns JWT tokens:
  ├── ID Token     — user identity claims (name, email, custom attributes)
  ├── Access Token — authorise access to your API (scopes)
  └── Refresh Token — obtain new ID/Access tokens without re-login
```

### Integrating User Pools with API Gateway / ALB
- **API Gateway Cognito authoriser**: API Gateway validates the JWT from the User Pool — no custom Lambda authoriser needed
- **ALB authentication**: ALB can redirect unauthenticated users to the Cognito Hosted UI, validate the token, and pass identity to the backend

### User Pool Lambda triggers
| Trigger | Use case |
|---|---|
| Pre sign-up | Validate or auto-confirm users |
| Post confirmation | Send welcome email, add to DynamoDB |
| Pre authentication | Block sign-in based on custom logic |
| Post authentication | Log sign-in events |
| Pre token generation | Add custom claims to JWT tokens |

## Cognito Identity Pools (Federated Identities)

An **Identity Pool** grants users **temporary AWS credentials** (via STS AssumeRoleWithWebIdentity) to access AWS services directly — S3, DynamoDB, API Gateway, etc.

### How Identity Pools work
```text
1. User authenticates with User Pool (or Google/Facebook/SAML)
         │
         ▼
2. App sends token to Identity Pool
         │
         ▼
3. Identity Pool calls STS → returns temporary AWS credentials
         │
         ▼
4. App uses credentials to call AWS directly (e.g., upload to S3)
```

### Identity sources
Identity Pools accept tokens from:
- Cognito User Pools
- Google, Facebook, Apple, Amazon
- SAML identity providers
- OpenID Connect (OIDC) providers
- **Unauthenticated (guest) access** — grant limited AWS access to anonymous users

### IAM roles for Identity Pools
Two IAM roles are associated with each Identity Pool:
- **Authenticated role** — permissions for logged-in users
- **Unauthenticated role** — limited permissions for guest users

You can also use **role-based access control** — map users to different IAM roles based on their User Pool group or token claims.

### User Pools vs Identity Pools

| | User Pools | Identity Pools |
|---|---|---|
| Purpose | Authenticate users (who you are) | Authorise AWS access (what you can do) |
| Output | JWT tokens (ID, Access, Refresh) | Temporary AWS credentials |
| Use case | Log in to your app, API authorisation | Access S3, DynamoDB, or other AWS services directly from client |
| Guest access | No | Yes (unauthenticated role) |

> **Common pattern:** User Pool (authentication) + Identity Pool (AWS access) used together — authenticate with the User Pool, exchange the JWT for AWS credentials via the Identity Pool.

## Cognito Advanced Features

### Adaptive authentication
Cognito analyses sign-in attempts for risk signals — new device, unusual location, impossible travel. For high-risk attempts it can require MFA or block the sign-in automatically.

### User Pool Groups
Organise users into groups (e.g., Admins, Editors, Viewers). Groups can be mapped to IAM roles in Identity Pools or checked in your application logic. The group membership is included in the JWT token as a claim.

### Hosted UI and custom domains
Cognito provides a hosted sign-in UI at a Cognito domain (`your-app.auth.us-east-1.amazoncognito.com`) or a custom domain. The Hosted UI supports OAuth 2.0 authorization code flow, implicit flow, and client credentials flow.

### App clients
Each application that authenticates against a User Pool has an **App Client** with its own client ID. You configure allowed OAuth flows, callback URLs, and which identity providers are available per client.

## Working with WAF and Cognito via boto3

In [ ]:
import boto3
import json

wafv2   = boto3.client('wafv2',          region_name='us-east-1')
cognito = boto3.client('cognito-idp',    region_name='us-east-1')   # User Pools
federated = boto3.client('cognito-identity', region_name='us-east-1')  # Identity Pools

In [ ]:
# Create an IP set to block malicious IPs
ip_set = wafv2.create_ip_set(
    Name='blocked-ips',
    Scope='REGIONAL',                    # REGIONAL for ALB/API GW; CLOUDFRONT for CF
    IPAddressVersion='IPV4',
    Addresses=['203.0.113.0/24', '198.51.100.42/32']
)
ip_set_arn = ip_set['Summary']['ARN']
print(f"IP set created: {ip_set_arn}")

In [ ]:
# Create a Web ACL with multiple rules
web_acl = wafv2.create_web_acl(
    Name='app-web-acl',
    Scope='REGIONAL',
    DefaultAction={'Allow': {}},          # allow by default; rules block specific traffic
    Rules=[
        {
            'Name': 'BlockBadIPs',
            'Priority': 1,
            'Action': {'Block': {}},
            'Statement': {
                'IPSetReferenceStatement': {'ARN': ip_set_arn}
            },
            'VisibilityConfig': {'SampledRequestsEnabled': True,
                                 'CloudWatchMetricsEnabled': True,
                                 'MetricName': 'BlockBadIPs'}
        },
        {
            'Name': 'RateLimit',
            'Priority': 2,
            'Action': {'Block': {}},
            'Statement': {
                'RateBasedStatement': {
                    'Limit': 2000,          # block if > 2000 requests per 5 minutes
                    'AggregateKeyType': 'IP'
                }
            },
            'VisibilityConfig': {'SampledRequestsEnabled': True,
                                 'CloudWatchMetricsEnabled': True,
                                 'MetricName': 'RateLimit'}
        },
        {
            'Name': 'AWSManagedCoreRules',
            'Priority': 3,
            'OverrideAction': {'None': {}},  # use the rule group's actions
            'Statement': {
                'ManagedRuleGroupStatement': {
                    'VendorName': 'AWS',
                    'Name': 'AWSManagedRulesCommonRuleSet'
                }
            },
            'VisibilityConfig': {'SampledRequestsEnabled': True,
                                 'CloudWatchMetricsEnabled': True,
                                 'MetricName': 'ManagedCoreRules'}
        }
    ],
    VisibilityConfig={'SampledRequestsEnabled': True,
                      'CloudWatchMetricsEnabled': True,
                      'MetricName': 'AppWebACL'}
)
web_acl_arn = web_acl['Summary']['ARN']
print(f"Web ACL created: {web_acl_arn}")

In [ ]:
# Create a Cognito User Pool
pool = cognito.create_user_pool(
    PoolName='MyAppUsers',
    Policies={
        'PasswordPolicy': {
            'MinimumLength': 12,
            'RequireUppercase': True,
            'RequireLowercase': True,
            'RequireNumbers': True,
            'RequireSymbols': True
        }
    },
    MfaConfiguration='OPTIONAL',         # OFF | OPTIONAL | ON
    AutoVerifiedAttributes=['email'],
    UsernameAttributes=['email'],          # sign in with email instead of username
    UserPoolTags={'Environment': 'production'}
)
user_pool_id = pool['UserPool']['Id']
print(f"User Pool ID: {user_pool_id}")

In [ ]:
# Create an App Client for a web application
app_client = cognito.create_user_pool_client(
    UserPoolId=user_pool_id,
    ClientName='web-app',
    GenerateSecret=False,                  # public client (browser/mobile)
    AllowedOAuthFlows=['code'],            # authorization code flow (most secure)
    AllowedOAuthScopes=['openid', 'email', 'profile'],
    CallbackURLs=['https://myapp.example.com/callback'],
    LogoutURLs=['https://myapp.example.com/logout'],
    AllowedOAuthFlowsUserPoolClient=True,
    SupportedIdentityProviders=['COGNITO', 'Google']
)
client_id = app_client['UserPoolClient']['ClientId']
print(f"App Client ID: {client_id}")

In [ ]:
# Create an Identity Pool linked to the User Pool
identity_pool = federated.create_identity_pool(
    IdentityPoolName='MyAppIdentityPool',
    AllowUnauthenticatedIdentities=False,
    CognitoIdentityProviders=[{
        'ProviderName': f'cognito-idp.us-east-1.amazonaws.com/{user_pool_id}',
        'ClientId': client_id,
        'ServerSideTokenCheck': True
    }]
)
identity_pool_id = identity_pool['IdentityPoolId']
print(f"Identity Pool ID: {identity_pool_id}")

# Attach IAM roles to the Identity Pool
federated.set_identity_pool_roles(
    IdentityPoolId=identity_pool_id,
    Roles={
        'authenticated':   'arn:aws:iam::123456789012:role/CognitoAuthRole',
        'unauthenticated': 'arn:aws:iam::123456789012:role/CognitoUnauthRole'
    }
)
print("IAM roles attached to Identity Pool")

## Summary

| Concept | Key Takeaway |
|---|---|
| Shield Standard | Free; always on; L3/L4 DDoS protection for all AWS customers |
| Shield Advanced | $3,000/month; L7 protection; SRT access; DDoS cost protection; detailed metrics |
| WAF | Layer 7 HTTP filtering; attaches to CloudFront, ALB, API Gateway, AppSync |
| Web ACL | Container for WAF rules; evaluated in priority order; default Allow or Block |
| WAF rate-based rule | Block IPs exceeding request rate threshold per 5-minute window |
| WAF managed rules | Pre-built AWS rule sets (OWASP, SQLi, Bot Control, Account Takeover) |
| WAF Count action | Match without blocking — use to test rules before enforcing |
| Firewall Manager | Centralise WAF + Shield policies across all Organisation accounts |
| Cognito User Pools | User directory; authentication; returns JWT tokens |
| User Pool JWT tokens | ID token (identity claims), Access token (API scopes), Refresh token |
| Social login | Google, Facebook, Apple, Amazon via User Pool federation |
| Enterprise federation | SAML 2.0 and OIDC providers (Azure AD, Okta) via User Pool |
| Lambda triggers | Pre/post sign-up, pre/post auth, pre token generation — custom logic at auth events |
| API Gateway + Cognito | Cognito authoriser validates JWT — no custom Lambda authoriser needed |
| ALB authentication | ALB redirects to Cognito Hosted UI; validates token before forwarding |
| Cognito Identity Pools | Exchange any identity token for temporary AWS credentials (STS) |
| Identity Pool use case | Client app needs direct AWS access — upload to S3, query DynamoDB |
| Authenticated / Unauth roles | Two IAM roles per Identity Pool; unauth = guest access |
| User Pool + Identity Pool | Common pattern: User Pool authenticates; Identity Pool grants AWS access |
| Adaptive authentication | Risk-based MFA challenge on suspicious sign-in attempts |